In [ ]:
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model we will use for semantic similarity.
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# Example question and its embedding vector.
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
v1

In [ ]:
# A candidate answer text and its embedding vector.
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
dv

In [ ]:
# Dot product gives a quick similarity score between question and answer.
v1.dot(dv)

np.float32(0.323324)

In [ ]:
# A different question, likely less related to the same answer.
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
# Compare the second question against the same answer vector.
v2.dot(dv)

np.float32(0.019730467)

In [ ]:
from ingest import load_faq_data

# Load FAQ documents (question/answer pairs) from the ingest pipeline.
documents = load_faq_data()

In [ ]:
# Join question and answer into one text per document for embedding.
texts = []

for doc in documents:
    text = doc["question"] + "    " + doc["answer"]
    texts.append(text)

In [ ]:
# Quick preview to verify the combined text looks right.
texts[:5]

["Course: When does the course start?    A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 "Course: What are the prerequisites for this course?    To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everyth

In [ ]:
# Progress bar helper for batch processing.
from tqdm.auto import tqdm

In [ ]:
# Encode texts in batches to avoid big memory spikes.
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i: i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/28 [00:00<?, ?it/s]

1368

In [ ]:
import numpy as np

# Convert list of vectors to a 2D NumPy array for fast math ops.
X = np.array(vectors)

In [ ]:
# Shape should be: (number_of_docs, embedding_dim).
X.shape

(1368, 384)

In [ ]:
# Encode the user query we want to search with.
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [ ]:
# Vectorized similarity: score every document against the query.
scores = X.dot(v_query)

In [ ]:
# Same scoring idea written explicitly with a loop.
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [ ]:
# Index of the single best match and its score.
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [ ]:
# Inspect the top document content.
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [ ]:
# Get indices of the 5 highest scores (unsorted ascending slice).
top5 = np.argsort(scores)[-5:]

In [ ]:
# Reverse to show best-to-worst among top 5.
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

# Same result in one line.
topk = np.argsort(scores)[-5:][::-1]

array([  2, 643, 925, 538,   7])

In [ ]:
# Print each top match with its score for manual inspection.
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [ ]:
# Scratch cell for quick checks/experiments.